# 行为树形式语义建模与 SMT 验证 —— AUP Learning Cloud 预演

本 notebook 在 AMD AUP Learning Cloud 的 JupyterLab 里**一键完成整个预演**：

1. 树结构与环境模拟（`run` 的等价物）
2. 显式状态空间探索（`explore`）
3. 有界模型检测与反例回放（`verify`）
4. 规模实验：状态爆炸曲线（`bench_scale.py` 的交互版）

> 运行方式：Kernel → Restart & Run All。要求已 `pip install z3-solver`，且当前目录是仓库根目录。

In [ ]:
import sys, time
sys.path.insert(0, ".")  # 仓库根目录

try:
    import matplotlib.pyplot as plt
    HAS_PLT = True
except ImportError:
    HAS_PLT = False
    print("(未安装 matplotlib，跳过绘图；pip install matplotlib 可启用图表)")
from btfsv.loader import load_json
from btfsv.fsm import build_machine
from btfsv.properties import Property, Atom, Event, Implies
from btfsv.tree import Action, Condition, Parallel, Sequence, Tree
from btfsv.z3_verify import verify
import z3
print("z3", z3.get_version_string(), "| 环境就绪")

## 1. 树结构与环境模拟

守卫 NPC：Selector（优先级清单）下挂"逃跑 / 治疗 / 巡逻"三个分支。环境 = 两个开关
（敌人可见？血量低？），每个 tick 问一遍"现在该干什么"。

In [ ]:
tree = load_json("examples/guard.json")
print(tree, "\n")

machine = build_machine(tree)
envs = [{"low_hp": False, "see_enemy": True}] * 6   # 敌人一直在
print(f"{'tick':>4} | {'环境':<26} | {'根状态':<4} | 事件")
for s in machine.simulate(tree, envs):
    ev = ",".join(s.events) if s.events else "-"
    print(f"{s.index:>4} | see_enemy={s.env['see_enemy']!s:<5} low_hp={s.env['low_hp']!s:<5} | {s.status.zh:<4} | {ev}")

## 2. 状态空间：状态 = 所有节点"书签"的合影

对有 starvation bug 的 `buggy.json` 做显式探索，打印每个可达状态的记忆快照。

In [ ]:
buggy = load_json("examples/buggy.json")
bm = build_machine(buggy)
print(f"可达状态 {bm.num_states} 个，转移边 {bm.num_transitions} 条（= 状态数 x 2^{len(bm.atoms)} 个环境）\n")
for i in range(bm.num_states):
    print(f"  s{i}  {bm.describe_state(buggy, i)}")

## 3. 有界模型检测：找到违反性质的环境序列

性质 P1：`G (see_enemy[持续 2 拍] -> 2 拍内 <flee>)`。z3 回答二选一：
**不存在**这样的环境（有界通过），或**存在**（给出反例并回放复核）。

In [ ]:
def show(result, tree):
    p = result.property
    print(f"[{p.name}]  LTL: {p.ltl}")
    print(f"结论: {result.status_zh}   求解 {result.stats.get('solve_seconds')}s")
    if result.steps:
        for s in result.steps:
            ev = ",".join(s.events) if s.events else "-"
            mark = "  <-- 反例窗口" if result.bad_index is not None and result.bad_index <= s.index <= result.bad_index + (p.within or 0) else ""
            print(f"  t={s.index}  see_enemy={s.env.get('see_enemy')} low_hp={s.env.get('low_hp')}  {s.status.zh}  {ev}{mark}")
        print(f"回放复核: {'通过' if result.confirmed else '未通过'}")

print("== buggy.json ==")
bm2 = build_machine(buggy)
show(verify(bm2, buggy.properties[0], tree=buggy), buggy)

print("\n== guard.json（修复版）==")
show(verify(machine, tree.properties[0], tree=tree), tree)

## 4. 规模实验：状态爆炸（这就是云端算力的需求来源）

树形：`Parallel(k=w)` 下挂 w 个 `Sequence[Condition(p_i), Action(d_i)]`。
理论预估 |S| ~ Π d_i（指数），|δ| = |S| x 2^w。观察单机峰值内存随 w 的增长。

In [ ]:
rows = []
for w in range(1, 8):
    d0 = 2
    atoms = tuple(f"p{i}" for i in range(w))
    branches = [Sequence(f"s{i}", [Condition(f"c{i}", atoms[i]), Action(f"a{i}", f"a{i}", d0 + i % 3)])
                for i in range(w)]
    tr = Tree(name=f"w{w}", root=Parallel("root", branches, k=w), atoms=atoms,
              actions=tuple(f"a{i}" for i in range(w)),
              properties=(Property.always("P", Implies(Event("a0"), Atom(atoms[0])), horizon=6),))
    t0 = time.perf_counter()
    try:
        m = build_machine(tr, state_limit=100_000, transition_limit=300_000)
        rows.append((w, m.num_states, m.num_transitions, time.perf_counter() - t0))
    except Exception as exc:
        rows.append((w, None, None, time.perf_counter() - t0))
        print(f"w={w}: 撞上限（{exc}）")
        break

ok = [(w, s, d, t) for w, s, d, t in rows if s]
print(f"{'w':>2} {'|S|':>9} {'|delta|':>11} {'抽取(s)':>8}")
for w, s_, d_, t_ in ok:
    print(f"{w:>2} {s_:>9,} {d_:>11,} {t_:>8.2f}")

if len(ok) >= 2:
    ws = [r[0] for r in ok]; ss = [r[1] for r in ok]
    if HAS_PLT:
        plt.figure(figsize=(7, 3.5))
        plt.semilogy(ws, ss, "o-", color="#2f6feb")
        plt.xlabel("并行分支数 w"); plt.ylabel("可达状态数 |S|（对数轴）")
        plt.title("状态爆炸：|S| ~ Pi(d_i)")
        plt.grid(True, which="both", alpha=0.3); plt.tight_layout(); plt.show()
    else:
        mx = max(ss)
        for w_, s_ in zip(ws, ss):
            print(f"w={w_} |" + "#" * max(1, round(40 * s_ / mx)) + f" {s_:,}")

## 结论与算力需求

* 状态数随并行宽度**指数增长**，转移边 = |S| x 2^w，BMC 编码规模 ~ |δ| x k；
* 单机内存/时间在 w=6~8 量级见顶（本节实测），外推 w=10 需 ~10^8 条边、约 25 GB 内存；
* 因此提出：按顶层子树分片 + 多机并行 BFS 的分布式探索方案 —— 云端集群（AUP Learning Cloud 的
  多节点 K3s 架构）正是该方案的预演平台。

下一步：`scripts/bench_scale.py --widths 1,2,3,4,5,6,7,8 --state-limit 500000` 跑更完整的扫描，
把 `build_cloud/bench.csv` 收进报告。